In [13]:
import pandas as pd
from top2vec import Top2Vec
import os
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import umap
import plotly.express as px
from pprint import pprint

In [14]:
stopwords=stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [15]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-stanza-lynch.feather')

In [ ]:
data.shape

In [16]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-stanza-lynch-whip.feather')

In [17]:
data.shape

(115645, 6)

In [ ]:
data.head(2)

In [18]:
data['article_stop'] = data['article'].str.lower().str.split().apply(lambda x: [word for word in x if word not in stopwords])

In [19]:
data['article_lemma'] = data['article_stop'].apply(lambda x: [WordNetLemmatizer().lemmatize(word) for word in x])

In [20]:
data['article_lemma_string']=data['article_lemma'].apply(lambda x: ' '.join(x))

In [ ]:
Counter(data['article_lemma'].explode()).most_common(10)

In [21]:
model=Top2Vec(documents=data['article_lemma_string'].tolist(), speed="learn", workers=4, embedding_model='distiluse-base-multilingual-cased')

2025-11-21 14:57:45,527 - top2vec - INFO - Pre-processing documents for training
/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2025-11-21 14:58:16,672 - top2vec - INFO - Downloading distiluse-base-multilingual-cased model
2025-11-21 14:58:18,235 - top2vec - INFO - Creating joint document/word embedding
2025-11-21 15:01:35,772 - top2vec - INFO - Creating lower dimension embedding of documents
2025-11-21 15:02:05,666 - top2vec - INFO - Finding dense areas of documents
/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed 

In [22]:
topic_sizes, topic_nums = model.get_topic_sizes()

In [23]:
print(len(topic_sizes), len(topic_nums)) #955 topics with distiluse-base-multilingual-cased

564 564


In [24]:
id_dic={}
topic_id={}
for element in zip(topic_nums, topic_sizes):
    documents, document_scores, document_ids = model.search_documents_by_topic(topic_num=element[0], num_docs=element[1])
    for score, id in zip(document_scores, document_ids):
        id_dic[id]=score
        topic_id[id]=element[0]

In [25]:
topic_words, word_scores, topic_scores = model.get_topics(len(topic_sizes))

In [26]:
df_words=pd.DataFrame(topic_words).transpose()
df_words.columns=topic_nums
df_words.columns = df_words.columns.astype(str)

In [41]:
df_words[['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15']].iloc[:10]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,negro,electorate,fortnight,prosecution,arrests,negro,twp,khan,khan,baseball,cemetery,congressman,legislature,electorate,legislature,sentencing
1,negroes,electoral,domingo,judicial,policia,negroes,tw,karakhan,karakhan,inning,funeral,senate,legislatures,khan,senate,penal
2,blacks,election,wednesday,sentencing,arrest,blacks,lw,shouted,clancy,innings,deceased,congressional,impeached,election,legislatures,judicial
3,arrests,elections,weekend,prosecutions,arresting,racial,dw,screamed,clan,championship,morgue,filibuster,legislative,elections,filibuster,justice
4,blackwell,reelected,miss,acquitted,policeman,nigger,twcnty,gandhi,emperor,rugby,died,congress,impeachment,electoral,legislative,legislative
5,arrest,constituency,madame,prosecutor,policemen,niggers,sw,habibullah,corrup,football,graveyard,caucus,senate,candidacy,congressional,legislature
6,arresting,legislature,mckinney,jury,police,black,cw,maharajah,corruption,batting,grob,presidential,governor,elected,congressman,impunity
7,nigger,elected,poned,convicted,sheriff,blackwell,pow,quash,dictatorial,pitcher,burial,parliamentary,gobernador,caucus,legislators,prosecute
8,niggers,voters,mckinley,litigation,criminals,slavery,hew,yelled,kians,teams,dead,senatorial,impeach,reelected,senatorial,jail
9,policia,balloting,thursday,tribunal,jailed,interracial,ow,shout,kurd,tournament,coffin,senator,gubernatorial,electing,legislator,penitentiary


In [28]:
data['topic_id']=data.index.map(topic_id) #map topic id to each document
data['topic_score']=data.index.map(id_dic) #map topic score to each document

In [42]:
Counter(data['topic_id']).most_common(20)

[(0.0, 486),
 (1.0, 479),
 (2.0, 424),
 (4.0, 379),
 (3.0, 378),
 (5.0, 336),
 (6.0, 319),
 (7.0, 281),
 (10.0, 254),
 (12.0, 253),
 (8.0, 247),
 (9.0, 246),
 (15.0, 243),
 (16.0, 227),
 (11.0, 224),
 (14.0, 221),
 (13.0, 219),
 (18.0, 216),
 (19.0, 199),
 (17.0, 196)]

In [ ]:
data[data['topic_id']==3]['article'].index

In [ ]:
data.to_csv('lynch-flog.csv', index=False)

In [ ]:
pprint(data[data['topic_id']==3]['article'].iloc[0])

In [ ]:
df_words.to_csv('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-stanza-lynch-topics-words.csv', index=False)

In [30]:
data[data['topic_id'].isna()].shape

(93021, 11)

In [ ]:
data[['article', 'topic_id', 'topic_score']].head(10)

In [32]:
data['vector']=model.document_vectors.tolist()

In [33]:
data.to_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-stanza-lynch-whip-top2vec.feather')

In [34]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-stanza-lynch-whip-top2vec.feather')

In [ ]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/american-stories/kkk-revival-top2vec.feather')

In [43]:
data.to_csv('lynch-whip.csv', index=False)

In [35]:
data['topic_id'].nunique()

564

In [36]:
notnull_data=data[data['topic_id'].notna()]

In [ ]:
topic_1=notnull_data[notnull_data['topic_id']==1]
topic_7=notnull_data[notnull_data['topic_id']==7]

In [ ]:
pprint(topic_7.iloc[12]['article'])

In [37]:
embedding = umap.UMAP().fit_transform(notnull_data['vector'].tolist()) #reduce dimensionality of vector representation

In [38]:
clusterable_embedding = umap.UMAP(
    n_neighbors=30,
    min_dist=0.0,
    n_components=3,
    random_state=42,
).fit_transform(embedding.data)

/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [39]:
fig = px.scatter_3d(
    notnull_data, 
    x=clusterable_embedding[:, 0], 
    y=clusterable_embedding[:, 1], 
    z=clusterable_embedding[:, 2], 
    color='topic_id',  # Specify the column for color
    hover_data=['topic_id']
)

fig.update_layout(
    autosize=False,
    width=1000,
    height=1000,
    title={"text": "The Vector Representation of KKK Articles with UMAP Dimension Reduction",
            "x": 0.5,
            "xanchor": "center"},
    scene=dict(
        aspectmode="manual",
        aspectratio=dict(x=1.3, y=1.3, z=1.3)  # ⬅ bigger plotting box
    ),
    scene_camera=dict(
        eye=dict(x=2, y=2, z=2)  # zoom out view
    )
)

fig.update_traces(marker=dict(size=3))

# fig.show()
fig.write_html("/Volumes/T7/chroniclingamerica/american-stories/interactive-plot-3d-new.html")
fig.write_image("/Volumes/T7/chroniclingamerica/american-stories/umap-3d-new.png")